In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import ltn
import pandas as pd
import numpy as np
import torch.nn as nn

In [2]:
base_model = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B" 

"""
alternative fine tuned version:
model_id = "abhi9ab/DeepSeek-R1-Distill-Llama-8B-finance-v1"
"""

tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    base_model,
    trust_remote_code=True,
    dtype=torch.bfloat16,
    device_map="auto"
)

# Make my own fine-tuned Lora-Adapter
# model = PeftModel.from_pretrained(model, "./your-trained-lora-adapter")

model = model.eval()
print("model loaded!")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

model loaded!


In [3]:
test_headlines = [
    "NVIDIA beats Q4 earnings expectations but warns of supply chain constraints in 2026."
]

In [4]:
for headline in test_headlines:
    prompt = f"<｜begin of sentence｜><｜User｜>Analyze the market sentiment of this news: {headline}<｜Assistant｜><｜thought｜>"
    
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=1024,
            temperature=0.1,
            top_p=0.95,
            pad_token_id=tokenizer.eos_token_id
        )
    
    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    clean_output = full_output.replace('Ġ', ' ').replace('Ċ', '\n').replace('pre', '')
    
    print(f"\nFINANCIAL ANALYSIS FOR: {headline}")
    print("-" * 50)
    
    if "Result" in clean_output:
        parts = clean_output.split("Result")
        print(f"REASONING: {parts[0].strip()}")
        print(f"FINAL SENTIMENT: Result {parts[1].strip()}")
    else:
        print(clean_output)
    
    print("="*50)


FINANCIAL ANALYSIS FOR: NVIDIA beats Q4 earnings expectations but warns of supply chain constraints in 2026.
--------------------------------------------------
<beginofsentence><｜User｜>Analyzethemarketsentimentofthisnews:NVIDIAbeatsQ4earningsexpectationsbutwarnsofsupplychainconstraintsin2026.<｜Assistant｜><thought>Alright, so I'm trying to figure out how to analyze the market sentiment of this news: "NVIDIA beats Q4 earnings expectations but warns of supply chain constraints in 20-26." Okay, let's break this down step by step.

First, I know that NVIDIA is a big company in the tech industry, especially in graphics cards and AI. Their earnings reports can really affect their stock price and investor sentiment. So, the news says they beat Q4 expectations. That's a positive sign because it means they're doing better than what people were expecting. Investors like consistency and when companies exceed forecasts, it can boost confidence.

But then there's a warning about supply chain constr

In [6]:
for headline in test_headlines:
    prompt = f"""
        You are a financial sentiment classifier.
        
        Analyze the headline and respond in EXACTLY this format:
        
        Sentiment: [Bullish, Bearish, or Neutral]
        Confidence: [number between 0 and 1]
        
        Rules:
        - Do NOT explain your reasoning
        - Output ONLY the two lines above
        - Be decisive
        
        Headline: {headline}
        """
            
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=1024,
            temperature=0.1,
            top_p=0.95,
            pad_token_id=tokenizer.eos_token_id
        )
    
    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    clean_output = full_output.replace('Ġ', ' ').replace('Ċ', '\n').replace('pre', '')
    
    print(f"\nFINANCIAL ANALYSIS FOR: {headline}")
    print("-" * 50)
    
    if "Result" in clean_output:
        parts = clean_output.split("Result")
        print(f"REASONING: {parts[0].strip()}")
        print(f"FINAL SENTIMENT: Result {parts[1].strip()}")
    else:
        print(clean_output)
    
    print("="*50)


FINANCIAL ANALYSIS FOR: NVIDIA beats Q4 earnings expectations but warns of supply chain constraints in 2026.
--------------------------------------------------
Youareafinancialsentimentclassifier.AnalyzetheheadlineandrespondinEXACTLYthisformat:Sentiment:[Bullish,Bearish,orNeutral]Confidence:[numberbetween0and1]Rules:-DoNOTexplainyourreasoning-OutputONLYthetwolinesabove-BedecisiveHeadline:NVIDIAbeatsQ4earningsexpectationsbutwarnsofsupplychainconstraintsin2026. NVIDIA'sQ4earningsexceedexpectations,buttheyalsohighlightedsupplychainconstraintsfor2026. So, the headline is about NVIDIA beating earnings expectations but also warning about supply chain constraints in 2026. Sentiment: Bullish. Confidence: 0.85. Rules: Do NOT explain your reasoning - Output ONLY the two lines above. Bedecisive
Alright, so I need to analyze the sentiment of the given headline regarding NVIDIA's Q4 earnings. The headline states that NVIDIA beat earnings expectations but also warned about supply chain constraints 

In [10]:
"""
deepseek model outputs a (batch_size, 4096) shaped tensor, which is fed into the neurosymbolic head

this tensor is a dense numerical vector representing the 'meaning' of the news headline.
"""

sentiment_head = torch.nn.Sequential(
    torch.nn.Linear(4096, 512),
    torch.nn.ReLU(),
    torch.nn.Linear(512, 1), 
    torch.nn.Sigmoid()
).to("cuda").to(torch.bfloat16)


class MaterialSentimentPredicate(torch.nn.Module):
    def forward(self, sentiment_output, magnitude):
        # if magnitude < 0.5, logic forces result to 0.5 (Neutral)
        is_material = (magnitude > 0.5).float()
        return sentiment_output * is_material + (0.5 * (1.0 - is_material))

is_bullish = ltn.Predicate(MaterialSentimentPredicate())


# freeze deepseek LLM weights
model.eval()
for param in model.parameters():
    param.requires_grad = False

# optimise just the head (neurosymbolic part)
optimizer = torch.optim.AdamW([
    {'params': sentiment_head.parameters(), 'lr': 1e-4},
], lr=1e-4)


# --- The Neurosymbolic Inference & Training Loop ---

def run_step(headline, magnitude_val, label_val):
    optimizer.zero_grad()
    
    # A. Neural Perception (DeepSeek)
    inputs = tokenizer(headline, return_tensors="pt").to("cuda")
    outputs = model(**inputs, output_hidden_states=True)
    
    # Get the embedding of the last token (the "thought" representation)
    # Shape: [1, 4096]
    last_hidden_state = outputs.hidden_states[-1][:, -1, :]
    
    # B. Generate Sentiment via Head
    raw_sentiment = sentiment_head(last_hidden_state)
    
    # C. Wrap in LTN (The Symbolic Bridge)
    s_var = ltn.Variable("sentiment", raw_sentiment)
    m_var = ltn.Variable("magnitude", torch.tensor([[magnitude_val]]).to("cuda").to(torch.bfloat16))
    
    # D. Logical Reasoning
    # This calls your MaterialSentimentPredicate
    prediction = is_bullish(s_var, m_var)
    
    # E. Calculate Loss based on the Logical Outcome
    target = torch.tensor([[label_val]]).to("cuda").to(torch.bfloat16)
    loss = torch.mean((prediction.value - target)**2)
    
    loss.backward()
    optimizer.step()
    return loss.item(), raw_sentiment.item(), prediction.value.item()

# Example Test
loss, neural_gut, final_dec = run_step("Market crashes as inflation spikes", 0.9, 0.0)
print(f"Neural Output: {neural_gut:.4f} | Logical Result: {final_dec:.4f}")

Neural Output: 0.5820 | Logical Result: 0.5820


In [11]:
test_scenarios = [
    # (Headline, Magnitude, Label, Scenario Name)
    ("The company reported a slight increase in office supply efficiency.", 0.1, 0.5, "Immaterial / Low Magnitude"),
    ("BREAKING: CEO arrested as company stock plummets 50% in pre-market trading.", 0.95, 0.0, "Material / Strong Bearish"),
    ("Federal Reserve announces surprise 50bps rate cut to stimulate growth.", 0.9, 1.0, "Material / Strong Bullish"),
    ("A local branch changed its weekend closing time by fifteen minutes.", 0.05, 0.5, "Noise / Near-Zero Magnitude")
]

print(f"{'Scenario':<30} | {'Neural Gut':<12} | {'Final Logic':<12} | {'Status'}")
print("-" * 80)

for headline, mag, label, name in test_scenarios:
    loss, neural_gut, final_dec = run_step(headline, mag, label)
    
    is_neutralized = "Forced Neutral" if (mag <= 0.5 and abs(final_dec - 0.5) < 1e-3) else "Neural Led"
    
    print(f"{name:<30} | {neural_gut:.4f}     | {final_dec:.4f}      | {is_neutralized}")

Scenario                       | Neural Gut   | Final Logic  | Status
--------------------------------------------------------------------------------
Immaterial / Low Magnitude     | 0.3770     | 0.5000      | Forced Neutral
Material / Strong Bearish      | 0.2217     | 0.2217      | Neural Led
Material / Strong Bullish      | 0.1660     | 0.1660      | Neural Led
Noise / Near-Zero Magnitude    | 0.3027     | 0.5000      | Forced Neutral
